# 🧠 IEEE ML Workshop — Hands-On Guide

## Build **iLert**: A Real-Time Drowsiness Detection System

Welcome to this hands-on workshop! By the end, you'll have built a working system that:

1. Captures live video from your webcam  
2. Detects your face using a **pre-trained machine-learning model** (MediaPipe Face Mesh)  
3. Tracks your **eye landmarks** to determine if your eyes are open or closed  
4. **Triggers an alarm** if you appear to be falling asleep  

---

### 📐 Architecture Overview

```
┌──────────────┐     ┌────────────────────┐     ┌──────────────────┐
│   Webcam     │────▶│  MediaPipe Face     │────▶│  Eye Ratio       │
│  (OpenCV)    │     │  Mesh (468 points)  │     │  Calculation     │
└──────────────┘     └────────────────────┘     └───────┬──────────┘
                                                        │
                                                        ▼
                                                ┌──────────────────┐
                                                │  Eyes closed for │
                                                │  > 1 second?     │
                                                └───────┬──────────┘
                                                   YES  │  NO
                                                   ▼    │  ▼
                                              🔊 ALARM  │  ✅ OK
```

### 🛠 Technologies Used

| Library | Purpose |
|---|---|
| **OpenCV** (`cv2`) | Camera capture & image processing |
| **MediaPipe** | Pre-trained face mesh ML model (468 landmarks) |
| **Pygame** | Audio playback for the alarm sound |
| **Python `time`** | Timing logic for the drowsiness threshold |

In [1]:
conda create -n ml_workshop python=3.11 -y
conda activate ml_workshop
pip install -r requirements.txt
#conda activate ml_workshop to activate environment after. conda deactivate for deactivation

SyntaxError: invalid syntax (4181769103.py, line 1)

---

## 📦 Section 1 — Environment Setup

Before we write any code, we need to install the required Python packages.

### ⚠️ Python Version Compatibility

This project requires **Python 3.9 – 3.12**. MediaPipe does **not** currently support Python 3.13+.  
Run the cell below to check your version:

In [1]:
import sys

major, minor = sys.version_info[:2]
print(f"Python version: {sys.version}")

if minor >= 13:
    print("\n❌ Python 3.13+ detected — MediaPipe is NOT compatible.")
    print("   Please use Python 3.9 – 3.12.")
    print("   Download from: https://www.python.org/downloads/")
elif minor < 9:
    print("\n❌ Python version is too old. Please upgrade to 3.9 – 3.12.")
else:
    print(f"\n✅ Python {major}.{minor} is compatible!")

Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]

✅ Python 3.11 is compatible!


### 📥 Install Dependencies

All required packages are listed in `requirements.txt`.  
Run the cell below **once** to install everything:

<details>
<summary>🔧 <strong>Click here if you don't have pip installed</strong></summary>

<br>

**pip** is Python's package manager. It usually comes pre-installed with Python.  
If you see a `pip: command not found` error, try these fixes:

**Option 1 — Use `python -m pip` instead:**
```
python -m pip install -r requirements.txt
```

**Option 2 — Install pip manually (Windows):**
1. Download [get-pip.py](https://bootstrap.pypa.io/get-pip.py)
2. Open a terminal in the folder where you downloaded it
3. Run: `python get-pip.py`

**Option 3 — Install pip manually (Mac / Linux):**
```bash
curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
python3 get-pip.py
```

**Option 4 — Use Conda (if you have Anaconda/Miniconda):**
```bash
conda install pip
```

After installing pip, re-run the install cell below.

</details>

In [6]:
# Install all dependencies from requirements.txt (run once)
!pip install -r requirements.txt

Now let's **import** the libraries and verify they loaded correctly.

In [2]:
import os
import time

import cv2                       # OpenCV — camera & image processing
import mediapipe as mp           # MediaPipe — face mesh ML model
import pygame                    # Pygame — plays our alarm sound

# Quick version check to confirm everything installed
print(f"OpenCV version:    {cv2.__version__}")
print(f"MediaPipe version: {mp.__version__}")
print(f"Pygame version:    {pygame.__version__}")
print("\n✅ All imports successful!")

pygame 2.6.1 (SDL 2.28.4, Python 3.11.15)
Hello from the pygame community. https://www.pygame.org/contribute.html
OpenCV version:    4.13.0
MediaPipe version: 0.10.14
Pygame version:    2.6.1

✅ All imports successful!


c:\Users\dagab\miniconda3\envs\ml_workshop\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


---

## 🧬 Section 2 — Understanding MediaPipe Face Mesh

### What is Face Mesh?

MediaPipe Face Mesh is a **pre-trained machine-learning model** developed by Google.  
It detects **468 3D landmarks** on a human face in real time — no training required on our end!

Each landmark is a point with:
- **`x`** — horizontal position (0.0 = left edge, 1.0 = right edge)  
- **`y`** — vertical position (0.0 = top edge, 1.0 = bottom edge)  
- **`z`** — depth (relative to the face)  

> 💡 The coordinates are **normalised** (0 to 1), so we multiply by the frame width/height to get pixel positions.

### Which Landmarks Do We Care About?

For drowsiness detection, we only need **4 landmarks** — the top and bottom of each eye:

| Eye | Top Landmark | Bottom Landmark |
|---|---|---|
| **Left Eye** | `159` | `145` |
| **Right Eye** | `386` | `374` |

When the eye is **open**, the vertical distance between top and bottom is large.  
When the eye is **closed**, that distance shrinks to nearly zero.

### Let's Initialise Face Mesh

In [3]:
# Create the Face Mesh model
# refine_landmarks=True gives us extra iris landmarks for higher accuracy
face_mesh = mp.solutions.face_mesh.FaceMesh(refine_landmarks=True)

print("✅ Face Mesh model loaded!")

✅ Face Mesh model loaded!


---

## 👁 Section 3 — Eye Landmark Detection

This is the **core ML concept** of the project. Let's break it down step by step.

### Step 1: Capture a Frame from the Webcam

OpenCV's `VideoCapture` gives us access to the webcam.  
Each call to `.read()` returns one frame (image).

In [4]:
# Open the default webcam (index 0)
cam = cv2.VideoCapture(0)

# Read a single frame
success, frame = cam.read()

if success:
    # Mirror the frame so it feels natural (like a mirror)
    frame = cv2.flip(frame, 1)
    
    # Get the frame dimensions
    frame_h, frame_w, _ = frame.shape
    print(f"✅ Captured a frame: {frame_w} x {frame_h} pixels")
else:
    print("❌ Could not capture frame. Is your webcam connected?")

# Release the camera (we'll reopen it when we need it)
cam.release()

✅ Captured a frame: 640 x 480 pixels


### Step 2: Run Face Mesh on the Frame

MediaPipe expects **RGB** images, but OpenCV captures in **BGR**.  
So we convert the colour space before processing.

In [5]:
# Re-capture a frame for processing
cam = cv2.VideoCapture(0)
success, frame = cam.read()

if success:
    frame = cv2.flip(frame, 1)
    frame_h, frame_w, _ = frame.shape
    
    # Convert BGR (OpenCV default) → RGB (MediaPipe expects this)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Run the face mesh model
    results = face_mesh.process(rgb_frame)
    
    # Check if any face was detected
    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark
        print(f"✅ Detected a face with {len(landmarks)} landmarks!")
    else:
        print("⚠️ No face detected. Make sure you're visible to the camera.")
else:
    print("❌ Could not capture frame.")

cam.release()

c:\Users\dagab\miniconda3\envs\ml_workshop\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✅ Detected a face with 478 landmarks!


### Step 3: Extract Eye Landmarks & Compute the Eye Ratio

The **eye ratio** is simply `bottom.y - top.y`:  
- **Open eye** → large positive number (e.g. 0.02 – 0.04)  
- **Closed eye** → very small number (e.g. < 0.004)

```
  Open Eye:          Closed Eye:
  
   ·  (top 159)       ·· (top ≈ bottom)
  / \                  ──
  \_/                 
   ·  (bottom 145)
   
  ratio ≈ 0.03        ratio ≈ 0.002
```

In [7]:
# Re-capture and process
cam = cv2.VideoCapture(0)
success, frame = cam.read()

if success:
    frame = cv2.flip(frame, 1)
    frame_h, frame_w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)
    
    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark
        
        # ── Left Eye ─────────────────────────────────
        left_top    = landmarks[159]   # Top of left eye
        left_bottom = landmarks[145]   # Bottom of left eye
        left_eye_ratio = left_bottom.y - left_top.y
        
        # ── Right Eye ────────────────────────────────
        right_top    = landmarks[386]  # Top of right eye
        right_bottom = landmarks[374]  # Bottom of right eye
        right_eye_ratio = right_bottom.y - right_top.y
        
        print(f"Left  eye ratio: {left_eye_ratio:.4f}")
        print(f"Right eye ratio: {right_eye_ratio:.4f}")
        print()
        
        # Try closing your eyes and running this cell again!
        if left_eye_ratio < 0.015:
            print("👁 Left eye:  CLOSED")
        else:
            print("👁 Left eye:  OPEN")
            
        if right_eye_ratio < 0.004:
            print("👁 Right eye: CLOSED")
        else:
            print("👁 Right eye: OPEN")
    else:
        print("⚠️ No face detected.")

cam.release()

Left  eye ratio: 0.0034
Right eye ratio: 0.0021

👁 Left eye:  CLOSED
👁 Right eye: CLOSED


---

## ⏱ Section 4 — Drowsiness Detection Algorithm

Detecting a **single** blink isn't enough — people blink all the time!  
We need to detect when someone's eyes stay closed **for too long**.

### The Algorithm

```
EVERY FRAME:
│
├─ Both eyes closed?
│   ├─ YES → Start a timer (if not already running)
│   │        └─ Timer > 1 second? → 🔊 TRIGGER ALARM
│   │
│   └─ NO  → Reset the timer
```

### Key Variables

| Variable | Purpose |
|---|---|
| `ALERT_THRESHOLD` | How many seconds before we trigger (default: 1) |
| `LEFT_EYE_THRESHOLD` | Ratio below which left eye is considered closed (0.015) |
| `RIGHT_EYE_THRESHOLD` | Ratio below which right eye is considered closed (0.004) |
| `both_eyes_closed_start_time` | Timestamp when both eyes first closed (or `None`) |

In [8]:
# ── Configuration ────────────────────────────────────────
ALERT_THRESHOLD      = 1       # Seconds of closed eyes before alarm
LEFT_EYE_THRESHOLD   = 0.015   # Left eye closure threshold
RIGHT_EYE_THRESHOLD  = 0.004   # Right eye closure threshold

print("Configuration set:")
print(f"  Alert after:          {ALERT_THRESHOLD} second(s)")
print(f"  Left eye threshold:   {LEFT_EYE_THRESHOLD}")
print(f"  Right eye threshold:  {RIGHT_EYE_THRESHOLD}")

Configuration set:
  Alert after:          1 second(s)
  Left eye threshold:   0.015
  Right eye threshold:  0.004


---

## 🔊 Section 5 — Alarm System

We use **Pygame's mixer** module to play an MP3 alarm sound.  
Make sure the file `alarm.mp3` is in the same folder as this notebook.

In [9]:
# Initialise the audio mixer
pygame.mixer.init()

# Build the path to alarm.mp3 (same folder as this notebook)
ALARM_PATH = os.path.join(os.getcwd(), "alarm.mp3")

# Verify the file exists
if os.path.exists(ALARM_PATH):
    print(f"✅ Alarm file found: {ALARM_PATH}")
else:
    print(f"❌ alarm.mp3 not found at: {ALARM_PATH}")
    print("   Make sure alarm.mp3 is in the same folder as this notebook.")


def play_alarm():
    """Load and play the alarm sound."""
    pygame.mixer.music.load(ALARM_PATH)
    pygame.mixer.music.play()
    print("🔊 Alarm triggered!")

✅ Alarm file found: c:\Users\dagab\iLert\alarm.mp3


Test the alarm by running the cell below:

In [10]:
# 🔊 Test the alarm (uncomment the line below and run)
# play_alarm()

---

## 🚀 Section 6 — Full Backend: Putting It All Together

Now we combine everything into one **main loop** that:

1. Reads frames from the webcam  
2. Runs face-mesh detection  
3. Computes eye ratios  
4. Tracks how long the eyes have been closed  
5. Plays the alarm if the threshold is exceeded  

> ⚠️ **Important:** This cell will open a camera window. Press **`q`** to quit.

In [11]:
# ============================================================
#  iLert — Full Detection Loop
#  Press 'q' in the camera window to quit.
# ============================================================

# ── Initialise ───────────────────────────────────────────
face_mesh = mp.solutions.face_mesh.FaceMesh(refine_landmarks=True)
cam = cv2.VideoCapture(0)  # Open default webcam

both_eyes_closed_start_time = None  # Timer for eye closure

print("Camera started. Press 'q' in the window to quit.")
print(f"Alert will trigger after {ALERT_THRESHOLD}s of closed eyes.\n")

# ── Main Loop ────────────────────────────────────────────
while True:
    success, frame = cam.read()
    if not success:
        continue  # Skip bad frames

    # Mirror the frame
    frame = cv2.flip(frame, 1)
    frame_h, frame_w, _ = frame.shape

    # Convert BGR → RGB for MediaPipe
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Run face mesh detection
    results = face_mesh.process(rgb_frame)

    # ── Process Detections ───────────────────────────────
    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark

        # Compute eye ratios
        left_eye_ratio  = landmarks[145].y - landmarks[159].y
        right_eye_ratio = landmarks[374].y - landmarks[386].y

        # Draw eye landmarks as yellow dots
        for lm in [landmarks[145], landmarks[159], landmarks[374], landmarks[386]]:
            px = int(lm.x * frame_w)
            py = int(lm.y * frame_h)
            cv2.circle(frame, (px, py), 3, (0, 255, 255), -1)

        # Check if both eyes are closed
        left_closed  = left_eye_ratio  < LEFT_EYE_THRESHOLD
        right_closed = right_eye_ratio < RIGHT_EYE_THRESHOLD
        both_closed  = left_closed and right_closed

        # ── Timer Logic ──────────────────────────────────
        if both_closed:
            if both_eyes_closed_start_time is None:
                # Eyes just closed — start timing
                both_eyes_closed_start_time = time.time()
            elif time.time() - both_eyes_closed_start_time >= ALERT_THRESHOLD:
                # Eyes closed long enough — ALERT!
                print("⚠️  ALERT: USER FELL ASLEEP!")
                cv2.putText(
                    frame, "WAKE UP!", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2
                )
                play_alarm()
                both_eyes_closed_start_time = None  # Reset
        else:
            # Eyes are open — reset timer
            both_eyes_closed_start_time = None

    # ── Display ──────────────────────────────────────────
    cv2.imshow("iLert — Drowsiness Monitor", frame)

    # Quit on 'q' key
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    time.sleep(0.05)  # Small delay to smooth frame rate

# ── Cleanup ──────────────────────────────────────────────
cam.release()
cv2.destroyAllWindows()
print("\n✅ Camera released. Session ended.")

Camera started. Press 'q' in the window to quit.
Alert will trigger after 1s of closed eyes.

⚠️  ALERT: USER FELL ASLEEP!
🔊 Alarm triggered!
⚠️  ALERT: USER FELL ASLEEP!
🔊 Alarm triggered!
⚠️  ALERT: USER FELL ASLEEP!
🔊 Alarm triggered!
⚠️  ALERT: USER FELL ASLEEP!
🔊 Alarm triggered!

✅ Camera released. Session ended.


---

## 🎨 Section 7 — Frontend (Pre-Built UI)

The code below is the **Kivy-based GUI** that wraps our detection logic into a
polished mobile-friendly app. This is provided **pre-built** — you don't need to
understand Kivy to complete this workshop.

The frontend:
- Displays a live camera preview via Kivy's `Camera` widget  
- Runs the **same detection logic** on a background thread  
- Shows status updates and triggers the alarm  

To run it, execute `python frontend.py` from your terminal.

In [12]:
# ============================================================
#  iLert Frontend — Pre-Built Kivy GUI
#  This code is shown for reference only.
#  Run it from the terminal:  python frontend.py
# ============================================================

# To view the full source, open frontend.py in your editor.
# Below is a simplified overview of how it works:

print("""
Frontend Architecture:
━━━━━━━━━━━━━━━━━━━━━

┌─────────────────────────────────┐
│          iLert (title)          │
│   Drowsiness Detection System   │
├─────────────────────────────────┤
│                                 │
│       [ Camera Preview ]        │
│        (Kivy Camera)            │
│                                 │
├─────────────────────────────────┤
│   Status: Monitoring active     │
├─────────────────────────────────┤
│     [ ▶ Start Monitoring ]      │
└─────────────────────────────────┘

The button toggles camera on/off.
Detection runs on a background thread.
When drowsiness is detected → alarm plays + status updates.
""")


Frontend Architecture:
━━━━━━━━━━━━━━━━━━━━━

┌─────────────────────────────────┐
│          iLert (title)          │
│   Drowsiness Detection System   │
├─────────────────────────────────┤
│                                 │
│       [ Camera Preview ]        │
│        (Kivy Camera)            │
│                                 │
├─────────────────────────────────┤
│   Status: Monitoring active     │
├─────────────────────────────────┤
│     [ ▶ Start Monitoring ]      │
└─────────────────────────────────┘

The button toggles camera on/off.
Detection runs on a background thread.
When drowsiness is detected → alarm plays + status updates.



---

## 🎯 Section 8 — Summary & Key Takeaways

### What You Learned Today

| Concept | What It Means |
|---|---|
| **Computer Vision** | Using code to "see" and interpret images from a camera |
| **Face Mesh** | A pre-trained ML model that finds 468 points on a face |
| **Landmark Detection** | Extracting specific facial points (eyes, nose, mouth) |
| **Threshold Classification** | Making decisions based on numerical cutoff values |
| **Real-Time Processing** | Analysing video frame by frame as it's captured |

### 🚀 Ideas for Improvement

Want to take this project further? Here are some ideas:

- **Yawn Detection** — Track mouth landmarks to detect yawning  
- **Head Pose Estimation** — Detect if the user's head is tilting / nodding off  
- **Logging** — Save timestamps of drowsiness events to a CSV file  
- **Better Thresholds** — Use the Eye Aspect Ratio (EAR) formula for more robust detection  
- **Mobile App** — Deploy to Android using Kivy + Buildozer  

### 📚 Resources

- [MediaPipe Face Mesh Documentation](https://developers.google.com/mediapipe/solutions/vision/face_landmarker)
- [OpenCV Python Documentation](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)
- [Pygame Documentation](https://www.pygame.org/docs/)
- [Face Landmark Index Map](https://github.com/google-ai-edge/mediapipe/blob/master/mediapipe/modules/face_geometry/data/canonical_face_model_uv_visualization.png)

---

### 🙏 Thank You!

Thanks for attending the **IEEE ML Workshop**!  
If you have questions, feel free to reach out to the workshop organisers.

Happy coding! 🎉